In [132]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import chi2
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

In [133]:
df = pd.read_csv("spaceship-titanic/train.csv")

selected_features_x = ["HomePlanet", "CryoSleep", "Cabin", "Destination", "Age", "VIP" , "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

cabin_split = df["Cabin"].astype(str).str.split("/", expand=True)
df["Deck"] = cabin_split[0]
df["CabinNum"] = pd.to_numeric(cabin_split[1], errors="coerce")
df["CabinType"] = cabin_split[2]

df["TOtal_Spend"] = df["RoomService"] + df["FoodCourt"] + df["ShoppingMall"] + df["Spa"] + df["VRDeck"]
df["IsAlone"] = (df["CabinNum"] == 0).astype(int)

df = df.drop(["Cabin", "PassengerId", "Name"], axis=1)

df.head()

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Deck,CabinNum,CabinType,TOtal_Spend,IsAlone
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False,B,0.0,P,0.0,1
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True,F,0.0,S,736.0,1
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False,A,0.0,S,10383.0,1
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False,A,0.0,S,5176.0,1
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True,F,1.0,S,1091.0,0


In [134]:
y = df["Transported"]
X = df.drop("Transported", axis=1)

numerical_features = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck", "CabinNum" , "TOtal_Spend"]
categorical_features = ["HomePlanet", "CryoSleep", "Destination", "VIP", "Deck", "CabinType" , "IsAlone"]

In [135]:
numerical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    # ("scaler", MinMaxScaler(feature_range=(0, 1))), 
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("to_str", FunctionTransformer(lambda Z: Z.astype(str), feature_names_out="one-to-one")),
    ("encoder", OneHotEncoder()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_preprocessor, numerical_features),
        ("cat", categorical_preprocessor, categorical_features),
    ],
    remainder="drop"
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("HistGradientBoostingClassifier", HistGradientBoostingClassifier(
        learning_rate=0.03,
        max_depth=5,
        min_samples_leaf=20,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        l2_regularization=0.1,
        random_state=42 
    ))
])

In [136]:

X_transformed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()
X_df = pd.DataFrame(X_transformed, columns=feature_names)

Y = y.astype(int)

In [137]:
chi2_scores, p_values = chi2(X_df, Y)

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Chi2 Score": chi2_scores,
    "p-value": p_values
}).sort_values(by="Chi2 Score", ascending=False)

selected_features_x = feature_importance[feature_importance["p-value"] < 0.09]["Feature"].tolist()
X_df = X_df[selected_features_x]
X_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 22 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   num__Spa                      8693 non-null   float64
 1   num__TOtal_Spend              8693 non-null   float64
 2   num__VRDeck                   8693 non-null   float64
 3   num__RoomService              8693 non-null   float64
 4   num__FoodCourt                8693 non-null   float64
 5   num__CabinNum                 8693 non-null   float64
 6   num__ShoppingMall             8693 non-null   float64
 7   cat__CryoSleep_True           8693 non-null   float64
 8   cat__CryoSleep_False          8693 non-null   float64
 9   num__Age                      8693 non-null   float64
 10  cat__HomePlanet_Europa        8693 non-null   float64
 11  cat__Deck_B                   8693 non-null   float64
 12  cat__HomePlanet_Earth         8693 non-null   float64
 13  cat

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

pipeline.fit(X_train, y_train)
accuracy = pipeline.score(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")

param_grid = {
    "HistGradientBoostingClassifier__learning_rate": [0.03, 0.05, 0.1],
    "HistGradientBoostingClassifier__max_depth": [None, 5, 10],
    "HistGradientBoostingClassifier__min_samples_leaf": [10, 20, 30],
    "HistGradientBoostingClassifier__max_iter": [300, 500, 800],
    "HistGradientBoostingClassifier__l2_regularization": [0.0, 0.1, 1.0]
}

grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

print(f"Best Hyperparameters: {grid_search.best_params_}")
best_pipeline = grid_search.best_estimator_
best_accuracy = best_pipeline.score(X_test, y_test)
print(f"Best Test Accuracy: {best_accuracy:.4f}")

best_params = grid_search.best_params_

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("HistGradientBoostingClassifier", HistGradientBoostingClassifier(learning_rate=best_params["HistGradientBoostingClassifier__learning_rate"],
        max_depth=best_params["HistGradientBoostingClassifier__max_depth"],
        min_samples_leaf=best_params["HistGradientBoostingClassifier__min_samples_leaf"],
        max_iter=best_params["HistGradientBoostingClassifier__max_iter"],
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        l2_regularization=best_params["HistGradientBoostingClassifier__l2_regularization"],
        random_state=42 
    ))
])

pipeline.fit(X_train, y_train)
accuracy = pipeline.score(X_test, y_test)
print(f"Test Accuracy after hyperparameter tuning: {accuracy:.4f}")

Test Accuracy after hyperparameter tuning: 0.8143
